In [ ]:
import os
import json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ---------------- 설정 ----------------
# 'asof'  : merge_asof backward. 2025 test 행이 가장 최근(2024) 트랙맨 값을 받는다.
#           학습/추론이 동일한 규칙을 쓰므로 원칙적으로 더 타당하다.
# 'exact' : (season, month) 정확 일치 + fillna(0). 900점 버전과 완전히 동일한 동작
#           (트랙맨에 2025가 없어 test에서는 전부 0이 된다).
TRACKMAN_MODE = 'asof'

N_SPLITS = 10                 # 5 -> 10 (각 fold가 90%를 학습, 평균 대상도 늘어 분산 감소)
SEEDS = [42, 202, 2024]       # seed 앙상블 (3개) -> 총 N_SPLITS * len(SEEDS) = 30개 모델
N_OPTUNA_TRIALS = 40          # 하이퍼파라미터 탐색 횟수

# --- 2026-08-18 추가 (2024 시즌 홀드아웃 3-seed 짝지어 검증 결과 반영) ---
# 조건부 투수통계: 기준선 대비 +20~27점 (3 seed 전부 우세, 분산도 ±18->±6로 감소)
USE_COND_STATS = True
# 재중심화: 11개 설정 전부에서 +11~22점 (평균 +18)
RECENTER = True
HOLDOUT_SEASON = 2024         # 오프셋 측정용 홀드아웃 시즌 (이 시즌은 학습에서 빼고 1회 측정)
N_HOLDOUT_FOLDS = 3
# 죽은 피처: asof_pitcher_n이 '경기내'가 아니라 '커리어 누적'이라 의도대로 동작하지 않음
#   is_long_relief 는 전체의 86%(이닝>1 중 97%)로 사실상 inning>1 과 동일,
#   is_strict_inherited_runner 는 0.05%로 상수, pitches_per_inning 은 커리어투구수/이닝.
#   (효과는 +4점 수준으로 미미하나 코드 정합성 차원에서 제거)
DEAD_FEATURES = ['is_long_relief', 'is_short_relief',
                 'is_strict_inherited_runner', 'pitches_per_inning']
print(f"TRACKMAN_MODE = {TRACKMAN_MODE} | N_SPLITS = {N_SPLITS} | SEEDS = {SEEDS}")


In [ ]:
STEPS_SRC = r"""
def step1_basic_features(df):
    df_proc = df.copy()
    df_proc['is_weekend_day_game'] = np.where(
        (df_proc['game_month'].isin([4, 5, 9, 10])) & (df_proc['game_dayofweek'].isin([5, 6])), 1.0, 0.0)
    df_proc['is_heat_wave_game'] = np.where(df_proc['game_month'].isin([7, 8]), 1.0, 0.0)
    return df_proc


def step2_pitcher_role_features(df):
    df_proc = df.copy()
    df_proc['is_pure_starter'] = np.where(df_proc['inning'] == 1, 1.0, 0.0)
    df_proc['is_long_relief'] = np.where(
        (df_proc['inning'] > 1) & (df_proc['asof_pitcher_n'] >= (df_proc['inning'] - 1) * 12), 1.0, 0.0)
    df_proc['is_short_relief'] = np.where(
        (df_proc['inning'] > 1) & (df_proc['asof_pitcher_n'] < (df_proc['inning'] - 1) * 12), 1.0, 0.0)
    return df_proc


def step3_matchup_features(df):
    df_proc = df.copy()
    if 'pitcher_hand' in df_proc.columns and 'batter_hand' in df_proc.columns:
        df_proc['is_same_hand'] = np.where(df_proc['pitcher_hand'] == df_proc['batter_hand'], 1.0, 0.0)
    return df_proc


def step4_refined_count_features(df):
    df_proc = df.copy()
    b, s = df_proc['balls_before'], df_proc['strikes_before']
    df_proc['is_first_pitch'] = np.where((b == 0) & (s == 0), 1.0, 0.0)
    df_proc['is_full_count'] = np.where((b == 3) & (s == 2), 1.0, 0.0)
    pitcher_ahead = ((b == 0) & (s == 1)) | ((b == 0) & (s == 2)) | ((b == 1) & (s == 2))
    batter_ahead = ((b == 1) & (s == 0)) | ((b == 2) & (s == 0)) | ((b == 3) & (s == 0)) | ((b == 2) & (s == 1)) | ((b == 3) & (s == 1))
    neutral = ((b == 1) & (s == 1)) | ((b == 2) & (s == 2))
    df_proc['count_advantage'] = np.select(
        [pitcher_ahead, batter_ahead, neutral], ['Pitcher', 'Batter', 'Neutral'], default='None')
    df_proc['is_waste_pitch_sit'] = np.where(((b == 0) & (s == 2)) | ((b == 1) & (s == 2)), 1.0, 0.0)
    df_proc['is_must_strike_sit'] = np.where(((b == 3) & (s == 0)) | ((b == 3) & (s == 1)), 1.0, 0.0)
    return df_proc


def step5_pitches_per_inning(df):
    df_proc = df.copy()
    df_proc['pitches_per_inning'] = df_proc['asof_pitcher_n'] / df_proc['inning'].clip(lower=1)
    return df_proc


def step6_combined_runner_features(df):
    df_proc = df.copy()
    df_proc['is_risp'] = df_proc['base_state'].astype(str).apply(
        lambda x: 1.0 if ('2' in x) or ('3' in x) else 0.0)
    df_proc['is_strict_inherited_runner'] = np.where(
        (df_proc['inning'] > 1) & (df_proc['asof_pitcher_n'] < 5) & (df_proc['num_runners_on'] > 0), 1.0, 0.0)
    df_proc['is_self_risp'] = np.where(
        (df_proc['asof_pitcher_n'] >= 15) & (df_proc['is_risp'] == 1.0), 1.0, 0.0)
    li_filled = df_proc['li'].fillna(0)
    df_proc['risp_pressure_index'] = df_proc['is_risp'] * li_filled
    df_proc['is_steal_threat_sit'] = np.where(
        (df_proc['runner_on_1b'] == 1) & (df_proc['runner_on_2b'] == 0)
        & (df_proc['score_diff_pitcher_team'].abs() <= 3), 1.0, 0.0)
    return df_proc


def step7_bayesian_smoothing(df, prior_mean=0.64):
    df_proc = df.copy()
    C = 50
    if 'asof_pitcher_success_rate' in df_proc.columns and 'asof_pitcher_n' in df_proc.columns:
        n = df_proc['asof_pitcher_n']
        curr = df_proc['asof_pitcher_success_rate']
        df_proc['smoothed_pitcher_success_rate'] = (n * curr + C * prior_mean) / (n + C)
    return df_proc


def step8_batter_toughness_features(df):
    df_proc = df.copy()
    if 'asof_batter_success_rate' in df_proc.columns and 'asof_batter_middle_rate' in df_proc.columns:
        df_proc['tough_batter_index'] = (1.0 - df_proc['asof_batter_success_rate']) * (1.0 - df_proc['asof_batter_middle_rate'])
    return df_proc


def step9_garbage_time_features(df):
    df_proc = df.copy()
    df_proc['is_garbage_time'] = np.where(df_proc['score_diff_pitcher_team'].abs() >= 7, 1.0, 0.0)
    df_proc['garbage_time_index'] = df_proc['score_diff_pitcher_team'].abs() / (10 - df_proc['inning']).clip(lower=1)
    return df_proc


def step10_recent_form_momentum(df):
    df_proc = df.copy()
    tc = ['asof_pitcher_prev1_game_success_rate',
          'asof_pitcher_prev3_game_success_rate',
          'asof_pitcher_prev5_game_success_rate']
    if all(c in df_proc.columns for c in tc):
        p1, p3, p5 = df_proc[tc[0]], df_proc[tc[1]], df_proc[tc[2]]
        df_proc['momentum_short'] = p1 - p3
        df_proc['momentum_mid'] = p1 - p5
        df_proc['is_heating_up'] = np.where((p1 > p3) & (p3 > p5), 1.0, 0.0)
        df_proc['is_cooling_down'] = np.where((p1 < p3) & (p3 < p5), 1.0, 0.0)
    return df_proc


def step11_veteran_and_pressure_features(df):
    df_proc = df.copy()
    df_proc['is_rookie'] = np.where(df_proc['asof_pitcher_n'] < 684, 1.0, 0.0)
    df_proc['is_veteran'] = np.where(df_proc['asof_pitcher_n'] > 3725, 1.0, 0.0)
    li_filled = df_proc['li'].fillna(0)
    df_proc['rookie_crisis_risk'] = df_proc['is_rookie'] * li_filled
    df_proc['veteran_clutch_ability'] = df_proc['is_veteran'] * li_filled
    return df_proc


def step12_first_pitch_tendency(df):
    df_proc = df.copy()
    if 'asof_pitcher_fastball_rate' in df_proc.columns and 'asof_pitcher_strike_rate' in df_proc.columns:
        if 'is_first_pitch' in df_proc.columns:
            df_proc['first_pitch_fastball_strike_idx'] = (
                df_proc['is_first_pitch'] * df_proc['asof_pitcher_fastball_rate'] * df_proc['asof_pitcher_strike_rate'])
    return df_proc


def step13_sac_fly_threat(df):
    df_proc = df.copy()
    is_3b = df_proc['base_state'].astype(str).apply(lambda x: 1.0 if '3' in x else 0.0)
    df_proc['is_sac_fly_threat'] = np.where(
        (is_3b == 1.0) & (df_proc['outs_before'] < 2)
        & (df_proc['score_diff_pitcher_team'].abs() <= 3), 1.0, 0.0)
    return df_proc


def step14_convert_to_category(df):
    df_proc = df.copy()
    original_cat_cols = ['pitcher_id', 'batter_id', 'pitcher_team_id', 'batter_team_id',
                         'pitcher_hand', 'batter_hand', 'base_state', 'stadium',
                         'pitch_name', 'top_bottom', 'game_type']
    created_cat_cols = ['is_weekend_day_game', 'is_heat_wave_game', 'is_pure_starter',
                        'is_long_relief', 'is_short_relief', 'is_same_hand', 'is_first_pitch',
                        'is_full_count', 'count_advantage', 'is_waste_pitch_sit',
                        'is_must_strike_sit', 'is_risp', 'is_strict_inherited_runner',
                        'is_self_risp', 'is_steal_threat_sit', 'is_sac_fly_threat',
                        'is_garbage_time', 'is_rookie', 'is_veteran',
                        'is_heating_up', 'is_cooling_down']
    all_cat_cols = [c for c in original_cat_cols + created_cat_cols if c in df_proc.columns]
    for c in all_cat_cols:
        df_proc[c] = df_proc[c].astype('category')
    return df_proc
"""

exec(STEPS_SRC)
print("step1~14 정의 완료")


In [ ]:
def step15_prep_trackman_data(trackman_df, pitcher_map_df):
    tm = pd.merge(trackman_df, pitcher_map_df[['pitcher_id', 'pitcher_trackman_id']],
                  on='pitcher_trackman_id', how='inner')
    b, s = tm['balls_before'], tm['strikes_before']
    p_ahead = ((b == 0) & (s == 1)) | ((b == 0) & (s == 2)) | ((b == 1) & (s == 2))
    b_ahead = ((b == 1) & (s == 0)) | ((b == 2) & (s == 0)) | ((b == 3) & (s == 0)) | ((b == 2) & (s == 1)) | ((b == 3) & (s == 1))
    neu = ((b == 1) & (s == 1)) | ((b == 2) & (s == 2))
    tm['count_advantage'] = np.select([p_ahead, b_ahead, neu],
                                       ['Pitcher', 'Batter', 'Neutral'], default='None')
    tm['pitch_group'] = tm['pitch_type_group'].astype(str).str.lower()
    return tm[tm['pitch_group'].isin(['fastball', 'breaking', 'offspeed'])].copy()


def step16_calc_expected_difficulty(tm):
    groups = ['fastball', 'breaking', 'offspeed']
    sit = tm.groupby(['season', 'game_month', 'pitcher_id', 'count_advantage', 'pitch_group']
                     ).size().unstack(fill_value=0).reset_index()
    for c in groups:
        if c not in sit.columns:
            sit[c] = 0
    sit = sit.sort_values(by=['pitcher_id', 'count_advantage', 'season', 'game_month'])
    g = sit.groupby(['pitcher_id', 'count_advantage'])
    sit['past_fb'] = g['fastball'].cumsum() - sit['fastball']
    sit['past_br'] = g['breaking'].cumsum() - sit['breaking']
    sit['past_off'] = g['offspeed'].cumsum() - sit['offspeed']
    tot = sit['past_fb'] + sit['past_br'] + sit['past_off']
    sit['past_total'] = tot
    sit['exp_fb_prob'] = np.where(tot > 0, sit['past_fb'] / tot, 0)
    sit['exp_br_prob'] = np.where(tot > 0, sit['past_br'] / tot, 0)
    sit['exp_off_prob'] = np.where(tot > 0, sit['past_off'] / tot, 0)

    dm = tm.groupby(['season', 'game_month', 'pitcher_id', 'pitch_group'])[['rel_height', 'rel_side']].std()
    dm['diff_score'] = dm['rel_height'] + dm['rel_side']
    dm = dm.reset_index()
    dp = dm.pivot_table(index=['season', 'game_month', 'pitcher_id'],
                        columns='pitch_group', values='diff_score', fill_value=np.nan).reset_index()
    for c in groups:
        if c not in dp.columns:
            dp[c] = 0
    dp = dp.sort_values(by=['pitcher_id', 'season', 'game_month'])
    gd = dp.groupby(['pitcher_id'])
    dp['past_fb_diff'] = gd['fastball'].transform(lambda x: x.shift(1).expanding().mean())
    dp['past_br_diff'] = gd['breaking'].transform(lambda x: x.shift(1).expanding().mean())
    dp['past_off_diff'] = gd['offspeed'].transform(lambda x: x.shift(1).expanding().mean())

    res = pd.merge(sit, dp, on=['season', 'game_month', 'pitcher_id'], how='left')
    res['expected_control_difficulty'] = (res['exp_fb_prob'] * res['past_fb_diff']
                                          + res['exp_br_prob'] * res['past_br_diff']
                                          + res['exp_off_prob'] * res['past_off_diff'])
    return res[['season', 'game_month', 'pitcher_id', 'count_advantage', 'expected_control_difficulty']]


def step17_calc_pitch_speed(tm):
    fb = tm[tm['pitch_group'] == 'fastball']
    sp = fb.groupby(['season', 'game_month', 'pitcher_id'])['rel_speed'].mean().reset_index()
    sp = sp.sort_values(by=['pitcher_id', 'season', 'game_month'])
    sp['past_fb_speed_mean'] = sp.groupby(['pitcher_id'])['rel_speed'].transform(
        lambda x: x.shift(1).expanding().mean())
    return sp[['season', 'game_month', 'pitcher_id', 'past_fb_speed_mean']]


def step18_calc_pitch_consistency_by_group(tm):
    groups = ['fastball', 'breaking', 'offspeed']
    metrics = ['rel_height_std', 'rel_side_std', 'extension_std',
               'spin_rate_std', 'vert_break_std', 'horz_break_std']
    cm = tm.groupby(['season', 'game_month', 'pitcher_id', 'pitch_group']).agg(
        rel_height_std=('rel_height', 'std'), rel_side_std=('rel_side', 'std'),
        extension_std=('extension', 'std'), spin_rate_std=('spin_rate', 'std'),
        vert_break_std=('induced_vert_break', 'std'), horz_break_std=('horz_break', 'std')
    ).reset_index()
    pv = cm.pivot_table(index=['season', 'game_month', 'pitcher_id'],
                        columns='pitch_group', values=metrics, fill_value=np.nan)
    pv.columns = [f"{grp}_{val}" for val, grp in pv.columns]
    pv = pv.reset_index()
    for pg in groups:
        for m in metrics:
            if f"{pg}_{m}" not in pv.columns:
                pv[f"{pg}_{m}"] = np.nan
    pv = pv.sort_values(by=['pitcher_id', 'season', 'game_month'])
    g = pv.groupby(['pitcher_id'])
    out_cols = ['season', 'game_month', 'pitcher_id']
    for pg in groups:
        for m in metrics:
            src, dst = f"{pg}_{m}", f"past_{pg}_{m}"
            pv[dst] = g[src].transform(lambda x: x.shift(1).expanding().mean())
            out_cols.append(dst)
    return pv[out_cols]


# ================= 조건부 투수통계 (2026-08-18 추가) =================
# 설계: 성공률이 매 시즌 단조 하락(.565->.486)하므로 원시 성공률을 그대로 쓰면 과거 시즌의
#       높은 수준이 그대로 섞여 들어온다. 그래서 '그 시즌 리그평균 대비 편차'로 디트렌드한 뒤
#       0(=리그평균)으로 shrink 하는 경험적 베이즈 방식을 쓴다.
#       표본이 적은 조합일수록 자동으로 0에 가까워지므로 콜드스타트도 자연히 처리된다.
# 검증: 2024 홀드아웃 3-seed 짝지어 비교에서 기준선 대비 +20(원본)/+27(재중심화)
COND_SPECS = [
    (['pitcher_id'],                                    200, 'cond_p'),
    (['pitcher_id', 'count_advantage'],                 100, 'cond_pc'),
    (['pitcher_id', 'batter_hand'],                     100, 'cond_ph'),
    (['pitcher_id', 'batter_hand', 'count_advantage'],   50, 'cond_phc'),
]


def _add_dev(df):
    """control_success 를 '그 시즌 리그평균 대비 편차'로 변환 (드리프트 제거)."""
    lg = df.groupby('season')['control_success'].mean()
    return df['control_success'] - df['season'].map(lg)


def build_cond_table(src, keys, C, name):
    g = src.groupby(keys, observed=True)['_dev'].agg(['sum', 'count']).reset_index()
    g[name] = g['sum'] / (g['count'] + C)          # 0(리그평균)으로 shrink
    return g[keys + [name]]


def attach_cond_features(df):
    """학습용: 각 행은 '그 시즌보다 과거' 데이터로만 인코딩 -> leak-free.
    (배포 시 2025 test 가 2019~2024 로 인코딩되는 것과 동일한 규칙)"""
    df = df.copy()
    df['_dev'] = _add_dev(df)
    seasons = sorted(df['season'].unique())
    for keys, C, name in COND_SPECS:
        col = np.full(len(df), np.nan)
        for s in seasons:
            past = df[df['season'] < s]
            if len(past) == 0:
                continue
            t = build_cond_table(past, keys, C, name).set_index(keys)[name]
            cur = (df['season'] == s).values
            sl = df.loc[cur, keys]
            idx = pd.MultiIndex.from_frame(sl) if len(keys) > 1 else pd.Index(sl[keys[0]])
            col[cur] = t.reindex(idx).values
        df[name] = col
        print(f"  {name}: 결측 {np.isnan(col).mean()*100:.1f}% (첫 시즌 + 신규투수)")
    return df.drop(columns=['_dev'])


def build_all_cond_tables(df):
    """추론용: 학습 전 시즌을 다 써서 만든 최종 룩업 테이블."""
    d = df.copy()
    d['_dev'] = _add_dev(d)
    return {name: build_cond_table(d, keys, C, name) for keys, C, name in COND_SPECS}


COND_COLS = [name for _, _, name in COND_SPECS]


In [ ]:
def run_full_pipeline(train_df, trackman_df, pitcher_map, trackman_mode='asof'):
    print(f"파이프라인 시작 (trackman_mode={trackman_mode})...")
    df_proc = train_df.copy()

    df_proc = step1_basic_features(df_proc)
    df_proc = step2_pitcher_role_features(df_proc)
    df_proc = step3_matchup_features(df_proc)
    df_proc = step4_refined_count_features(df_proc)
    df_proc = step5_pitches_per_inning(df_proc)
    df_proc = step6_combined_runner_features(df_proc)

    prior_mean = float(df_proc['asof_pitcher_success_rate'].mean())
    print(f"  prior_mean = {prior_mean:.6f}")

    df_proc = step7_bayesian_smoothing(df_proc, prior_mean=prior_mean)
    df_proc = step8_batter_toughness_features(df_proc)
    df_proc = step9_garbage_time_features(df_proc)
    df_proc = step10_recent_form_momentum(df_proc)
    df_proc = step11_veteran_and_pressure_features(df_proc)
    df_proc = step12_first_pitch_tendency(df_proc)
    df_proc = step13_sac_fly_threat(df_proc)

    if 'count_advantage' not in df_proc.columns:
        b, s = df_proc['balls_before'], df_proc['strikes_before']
        p_ahead = ((b == 0) & (s == 1)) | ((b == 0) & (s == 2)) | ((b == 1) & (s == 2))
        b_ahead = ((b == 1) & (s == 0)) | ((b == 2) & (s == 0)) | ((b == 3) & (s == 0)) | ((b == 2) & (s == 1)) | ((b == 3) & (s == 1))
        neu = ((b == 1) & (s == 1)) | ((b == 2) & (s == 2))
        df_proc['count_advantage'] = np.select([p_ahead, b_ahead, neu],
                                                ['Pitcher', 'Batter', 'Neutral'], default='None')

    tm_base = step15_prep_trackman_data(trackman_df, pitcher_map)
    feat_diff = step16_calc_expected_difficulty(tm_base)
    feat_speed = step17_calc_pitch_speed(tm_base)
    feat_rp = step18_calc_pitch_consistency_by_group(tm_base)
    rp_value_cols = [c for c in feat_rp.columns if c.startswith('past_')]

    if trackman_mode == 'asof':
        for f in [feat_diff, feat_speed, feat_rp]:
            f['time_idx'] = f['season'] * 100 + f['game_month']
            f.sort_values('time_idx', inplace=True)
        df_proc['time_idx'] = df_proc['season'] * 100 + df_proc['game_month']
        df_proc = df_proc.sort_values('time_idx')

        df_proc = pd.merge_asof(
            df_proc,
            feat_diff[['time_idx', 'pitcher_id', 'count_advantage', 'expected_control_difficulty']],
            on='time_idx', by=['pitcher_id', 'count_advantage'], direction='backward')
        df_proc = pd.merge_asof(
            df_proc, feat_speed[['time_idx', 'pitcher_id', 'past_fb_speed_mean']],
            on='time_idx', by='pitcher_id', direction='backward')
        df_proc = pd.merge_asof(
            df_proc, feat_rp[['time_idx', 'pitcher_id'] + rp_value_cols],
            on='time_idx', by='pitcher_id', direction='backward')
        df_proc = df_proc.drop(columns=['time_idx'])
    else:
        df_proc = pd.merge(df_proc, feat_diff,
                           on=['season', 'game_month', 'pitcher_id', 'count_advantage'], how='left')
        df_proc = pd.merge(df_proc, feat_speed,
                           on=['season', 'game_month', 'pitcher_id'], how='left')
        df_proc = pd.merge(df_proc, feat_rp,
                           on=['season', 'game_month', 'pitcher_id'], how='left')
        for c in ['expected_control_difficulty', 'past_fb_speed_mean'] + rp_value_cols:
            if c in df_proc.columns:
                df_proc[c] = df_proc[c].fillna(0)

    # 조건부 투수통계 (step14 이전에 붙여야 함: pitcher_id/count_advantage 가 아직 원시 dtype)
    cond_tables = {}
    if USE_COND_STATS:
        print("조건부 투수통계 생성...")
        df_proc = attach_cond_features(df_proc)
        cond_tables = build_all_cond_tables(df_proc)   # 추론용 최종 테이블(전 시즌)

    df_proc = step14_convert_to_category(df_proc)
    print("파이프라인 완료.")
    return df_proc.reset_index(drop=True), prior_mean, feat_diff, feat_speed, feat_rp, cond_tables


In [ ]:
DATA_DIR = "/kaggle/input/datasets/homekeggle/aimers/open/data"
MAP_PATH = "/kaggle/input/datasets/homekeggle/aimers/pitcher_id_mapping.csv"

df_train = pd.read_csv(f"{DATA_DIR}/train.csv")
df_trackman = pd.read_csv(f"{DATA_DIR}/trackman_history.csv")
pitcher_id_mapping = pd.read_csv(MAP_PATH)
print("train:", df_train.shape, "| trackman:", df_trackman.shape)


In [ ]:
df_processed, PRIOR_MEAN, feat_diff, feat_speed, feat_rp, cond_tables = run_full_pipeline(
    df_train, df_trackman, pitcher_id_mapping, trackman_mode=TRACKMAN_MODE)
print("df_processed:", df_processed.shape)


In [ ]:
os.makedirs("model", exist_ok=True)

with open("model/train_constants.json", "w") as f:
    json.dump({"prior_mean": PRIOR_MEAN, "trackman_mode": TRACKMAN_MODE}, f)
print(f"train_constants.json  prior_mean={PRIOR_MEAN:.6f}  trackman_mode={TRACKMAN_MODE}")

# 트랙맨 테이블은 step16~18 출력 그대로 저장 (dropna/dedup 금지)
_rp = [c for c in feat_rp.columns if c.startswith('past_')]
_diff_cols = ['season', 'game_month', 'pitcher_id', 'count_advantage', 'expected_control_difficulty']
_speed_cols = ['season', 'game_month', 'pitcher_id', 'past_fb_speed_mean']
_rp_cols = ['season', 'game_month', 'pitcher_id'] + _rp
if TRACKMAN_MODE == 'asof':
    for f_, extra in [(feat_diff, _diff_cols), (feat_speed, _speed_cols), (feat_rp, _rp_cols)]:
        if 'time_idx' not in f_.columns:
            f_['time_idx'] = f_['season'] * 100 + f_['game_month']
    _diff_cols = ['time_idx'] + _diff_cols
    _speed_cols = ['time_idx'] + _speed_cols
    _rp_cols = ['time_idx'] + _rp_cols

feat_diff[_diff_cols].to_csv("model/feat_diff.csv", index=False)
feat_speed[_speed_cols].to_csv("model/feat_speed.csv", index=False)
feat_rp[_rp_cols].to_csv("model/feat_rp.csv", index=False)
print(f"feat_diff {len(feat_diff):,} / feat_speed {len(feat_speed):,} / feat_rp {len(feat_rp):,}")

# 'None' 라운드트립 검증: 0-0/3-2 카운트를 뜻하는 실제 문자열인데
# pd.read_csv 기본 설정은 NaN으로 읽어버려 merge가 전량 실패한다.
_NA = ['', 'NaN', 'nan', 'NULL', 'null', 'NA', 'N/A', 'n/a']
_chk = pd.read_csv("model/feat_diff.csv", keep_default_na=False, na_values=_NA)
_n_none = (_chk['count_advantage'].astype(str) == 'None').sum()
_bad = pd.read_csv("model/feat_diff.csv")['count_advantage'].isna().sum()
print(f"\n'None' 행 {_n_none:,}개 — 기본 read_csv로는 {_bad:,}개가 NaN이 됨 (script.py는 na_values 명시)")
assert _n_none > 0, "'None' 값이 사라졌습니다"

# 조건부 투수통계 테이블 저장 (추론에서 룩업)
# count_advantage 의 'None' 은 0-0/3-2 를 뜻하는 실제 문자열이라 라운드트립 검증 필수 (4-3)
for _name, _tbl in cond_tables.items():
    _tbl.to_csv(f"model/{_name}.csv", index=False)
    print(f"{_name}.csv  {len(_tbl):,}행")
if 'cond_phc' in cond_tables:
    _c = pd.read_csv("model/cond_phc.csv", keep_default_na=False, na_values=_NA)
    assert (_c['count_advantage'].astype(str) == 'None').sum() > 0, "'None' 유실"
    print("조건부 테이블 'None' 라운드트립 OK")


In [ ]:
# Optuna 생략: 직전 실행(v4)에서 찾은 파라미터를 그대로 사용한다.
# 목적은 recenter_offset 상수 하나를 얻는 것뿐이며, 모델 30개는 이미 학습·회수 완료.
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from catboost import CatBoostClassifier

target_col = 'control_success'
drop_cols = [target_col, 'row_id', 'pitcher_id', 'batter_id', 'time_idx'] + DEAD_FEATURES
feature_cols = [c for c in df_processed.columns if c not in drop_cols]
X_full = df_processed[feature_cols].copy()
y_full = df_processed[target_col].copy()
for col in [c for c in X_full.columns if X_full[c].dtype.name in ['category', 'object']]:
    X_full[col] = X_full[col].astype(str).astype('category')
cat_features = [c for c in X_full.columns if X_full[c].dtype.name == 'category']
X, y = X_full, y_full

BEST_PARAMS = {
    "learning_rate": 0.022831883708228414,
    "depth": 8,
    "l2_leaf_reg": 8.552069332567962,
    "bagging_temperature": 0.05636104060100738,
    "random_strength": 0.7731135614050382,
    "iterations": 1000,
    "eval_metric": "Logloss",
    "task_type": "GPU",
    "early_stopping_rounds": 50
}
BEST_PARAMS["cat_features"] = cat_features

# v4 산출물과 피처 구성이 같은지 확인 (다르면 오프셋을 그 모델에 쓸 수 없다)
_ref = json.load(open("/kaggle/input/datasets/homekeggle/aimers/selected_features_v4.json")) \
       if os.path.exists("/kaggle/input/datasets/homekeggle/aimers/selected_features_v4.json") else None
if _ref is not None:
    assert list(feature_cols) == _ref, "피처 구성이 v4와 다릅니다!"
    print("피처 구성 v4와 일치 확인")
print(f"피처 {len(feature_cols)}개 (범주형 {len(cat_features)}개)")


def extract_isotonic(cv_obj):
    cc = cv_obj.calibrated_classifiers_[0]
    if hasattr(cc, 'calibrators'):
        return cc.calibrators[0]
    if hasattr(cc, 'calibrators_'):
        return cc.calibrators_[0]
    raise AttributeError("보정기를 찾을 수 없습니다.")


In [ ]:
# 홀드아웃(=2023까지 학습 -> 2024 예측)으로 로짓 오프셋을 측정해 상수로 고정한다.
# X, y, cat_features, BEST_PARAMS, extract_isotonic 은 Cell 6a/6b 에서 정의됨.

def solve_logit_offset(p, target):
    """평균 예측이 target 이 되게 하는 로짓 공간 상수 시프트."""
    q = np.clip(p, 1e-6, 1 - 1e-6)
    lo = np.log(q / (1 - q))
    off = 0.0
    for _ in range(300):
        cur = 1.0 / (1.0 + np.exp(-(lo + off)))
        err = cur.mean() - target
        if abs(err) < 1e-9:
            break
        off -= err * 4.0
    return float(off)


RECENTER_OFFSET = 0.0
if RECENTER:
    _tr_m = (df_processed['season'] <= HOLDOUT_SEASON - 1).to_numpy()
    _va_m = (df_processed['season'] == HOLDOUT_SEASON).to_numpy()
    _Xh, _yh = X[_tr_m], y[_tr_m]
    _Xv, _yv = X[_va_m], y[_va_m]
    print(f"오프셋 측정: 학습 {len(_Xh):,}행(~{HOLDOUT_SEASON-1}) -> 검증 {len(_Xv):,}행({HOLDOUT_SEASON})")

    _skf = StratifiedKFold(n_splits=N_HOLDOUT_FOLDS, shuffle=True, random_state=SEEDS[0])
    _ps = []
    for _f, (_ti, _vi) in enumerate(_skf.split(_Xh, _yh)):
        _p = dict(BEST_PARAMS); _p["random_seed"] = SEEDS[0]
        _m = CatBoostClassifier(**_p)
        _m.fit(_Xh.iloc[_ti], _yh.iloc[_ti], eval_set=(_Xh.iloc[_vi], _yh.iloc[_vi]), verbose=0)
        _c = CalibratedClassifierCV(_m, method='isotonic', cv='prefit')
        _c.fit(_Xh.iloc[_vi], _yh.iloc[_vi])
        _ps.append(extract_isotonic(_c).predict(_m.predict_proba(_Xv)[:, 1]))
        print(f"  홀드아웃 fold {_f+1}/{N_HOLDOUT_FOLDS} 완료")

    _ph = np.mean(_ps, axis=0)
    _actual = float(_yv.mean())
    RECENTER_OFFSET = solve_logit_offset(_ph, _actual)

    _naive = _actual * (1 - _actual)
    _sk = lambda q: (1 - ((np.clip(q, 1e-6, 1-1e-6) - _yv.to_numpy()) ** 2).mean() / _naive) * 100000
    _qq = np.clip(_ph, 1e-6, 1-1e-6)
    _after = 1.0 / (1.0 + np.exp(-(np.log(_qq/(1-_qq)) + RECENTER_OFFSET)))
    print(f"\n  {HOLDOUT_SEASON} 실제평균={_actual:.4f} | 보정전 평균예측={_ph.mean():.4f}")
    print(f"  로짓 오프셋 = {RECENTER_OFFSET:+.4f}")
    print(f"  홀드아웃 환산점수: 보정전 {_sk(_ph):,.0f} -> 보정후 {_sk(_after):,.0f}  ({_sk(_after)-_sk(_ph):+,.0f})")

# train_constants.json 갱신 (script.py 가 이 상수를 그대로 더한다)
with open("model/train_constants.json", "w") as f:
    json.dump({"prior_mean": PRIOR_MEAN, "trackman_mode": TRACKMAN_MODE,
               "recenter_offset": RECENTER_OFFSET}, f)
print(f"\ntrain_constants.json 저장: recenter_offset={RECENTER_OFFSET:+.4f}")
